# 03b — Fusion retrain LOCAL (для получения ftmff_catboost.cbm)**Цель:** переобучить Fusion_team локально на маке, сохранить как `ftmff_catboost.cbm` для отправки Диане.**НЕ ЛОМАЕМ оригинал** — это копия `03_multimodal_colab.ipynb`, адаптированная под локальный запуск.Время выполнения: ~5-10 минут (text e5 уже посчитан, CLIP уже посчитан, нужно только PCA + CatBoost).Все пути локальные — никакого Colab не требуется.## Что делается1. Загрузка `ozon_train.csv` (локально с Desktop)2. Загрузка CLIP embeddings (локально с Desktop)3. Загрузка cached text e5 embeddings (локально из team_runs/embeddings/)4. PCA-25 на CLIP и e5 → 92-dim Fusion features5. CatBoost обучение с теми же гиперпараметрами (seed=42)6. **`model.save_model('ftmff_catboost.cbm')`** — главная новинка7. Sanity check: probas из переобученной модели сравниваются с `test_proba_fusion_team.npy`

In [ ]:
# ---- Setup (LOCAL) ------------------------------------------------------import os, sys, json, time, hashlibimport numpy as npimport pandas as pdfrom pathlib import PathROOT = Path('/Users/sofya/women-in-data-thesis/.claude/worktrees/dazzling-vaughan')TR = ROOT / 'team_runs'DATA = '/Users/sofya/Desktop/diplomahse/ozon_train.csv'CLIP = '/Users/sofya/Desktop/diplomahse/clip_embeddings.parquet'E5_PATH = TR / 'embeddings' / 'text_e5_small.parquet'SPLITS = TR / 'splits'PROBA = TR / 'proba'ARTIFACTS = TR / 'artifacts'ARTIFACTS.mkdir(parents=True, exist_ok=True)# Sanity: all input files existfor p in [DATA, CLIP, E5_PATH, SPLITS/'team_train_idx.npy', SPLITS/'team_val_idx.npy',          SPLITS/'team_test_idx.npy', SPLITS/'y_test.npy', TR/'notebooks'/'utils.py']:    assert Path(p).exists(), f'MISSING: {p}'sys.path.insert(0, str(TR / 'notebooks'))from utils import normalize_emb_df, evaluate, r_at_p90, avg_precision_metricprint('Setup OK, all files found')

In [ ]:
# ---- Load data + splits -------------------------------------------------df = pd.read_csv(DATA, encoding='utf-8')df['ItemID'] = df['ItemID'].astype('int64')train_idx = np.load(SPLITS / 'team_train_idx.npy')val_idx   = np.load(SPLITS / 'team_val_idx.npy')test_idx  = np.load(SPLITS / 'team_test_idx.npy')y_test_saved = np.load(SPLITS / 'y_test.npy')y_train = df.iloc[train_idx]['resolution'].astype('int8').valuesy_val   = df.iloc[val_idx]['resolution'].astype('int8').valuesy_test  = df.iloc[test_idx]['resolution'].astype('int8').valuesassert (y_test == y_test_saved).all()spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))CATBOOST_BASE = dict(    iterations=1000, depth=6, learning_rate=0.05,    eval_metric='AUC', scale_pos_weight=spw,    early_stopping_rounds=50, random_seed=42, verbose=100,)print(f'sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')print(f'scale_pos_weight = {spw:.4f}')

In [ ]:
# ---- Load cached embeddings (CLIP + text e5) ----------------------------clip_raw = pd.read_parquet(CLIP)clip_norm = normalize_emb_df(clip_raw, expected_dim=512, prefix='clip', id_col='ItemID')print(f'CLIP normalized: {clip_norm.shape}')del clip_rawtext_emb = pd.read_parquet(E5_PATH)assert text_emb.shape[1] - 1 == 384, f'expected 384 dim, got {text_emb.shape[1]-1}'print(f'text e5: {text_emb.shape}')

In [ ]:
# ---- PCA-25 on both modalities + tabular prep ---------------------------from sklearn.decomposition import PCAfrom catboost import CatBoostClassifierMY_FEATURES = [c for c in df.columns if c not in ['resolution', 'ItemID', 'SellerID']]MY_CATS = df[MY_FEATURES].select_dtypes(include=['object', 'string', 'category']).columns.tolist()print(f'MY_FEATURES={len(MY_FEATURES)}, MY_CATS={MY_CATS}')def prep_tab(idx):    X = df.iloc[idx][MY_FEATURES].copy()    nums = [c for c in MY_FEATURES if c not in MY_CATS]    if nums:        X[nums] = X[nums].fillna(0)    for c in MY_CATS:        X[c] = X[c].fillna('nan').astype(str)    return X.reset_index(drop=True)def merge_emb(idx, emb_full, dim):    flat = [c for c in emb_full.columns if c != 'ItemID']    assert len(flat) == dim    merged = df.iloc[idx][['ItemID']].astype({'ItemID': 'int64'}).reset_index(drop=True).merge(        emb_full, on='ItemID', how='left'    )    return merged[flat].fillna(0).values.astype('float32')# CLIP-PCA-25et_c = merge_emb(train_idx, clip_norm, 512)ev_c = merge_emb(val_idx,   clip_norm, 512)es_c = merge_emb(test_idx,  clip_norm, 512)pca_c = PCA(n_components=25, random_state=42).fit(et_c)pt_c, pv_c, ps_c = pca_c.transform(et_c), pca_c.transform(ev_c), pca_c.transform(es_c)print(f'CLIP PCA-25 var={pca_c.explained_variance_ratio_.sum():.4f}')# Text-PCA-25et_t = merge_emb(train_idx, text_emb, 384)ev_t = merge_emb(val_idx,   text_emb, 384)es_t = merge_emb(test_idx,  text_emb, 384)pca_t = PCA(n_components=25, random_state=42).fit(et_t)pt_t, pv_t, ps_t = pca_t.transform(et_t), pca_t.transform(ev_t), pca_t.transform(es_t)print(f'Text PCA-25 var={pca_t.explained_variance_ratio_.sum():.4f}')del clip_norm, text_emb, et_c, ev_c, es_c, et_t, ev_t, es_t

In [ ]:
# ---- Build Fusion feature matrix (92-dim) -------------------------------Xt = prep_tab(train_idx)Xv = prep_tab(val_idx)Xs = prep_tab(test_idx)clip_cols = [f'clip_pca_{i}' for i in range(25)]text_cols = [f't_pca_{i}'    for i in range(25)]Xt = pd.concat([Xt, pd.DataFrame(pt_c, columns=clip_cols), pd.DataFrame(pt_t, columns=text_cols)], axis=1)Xv = pd.concat([Xv, pd.DataFrame(pv_c, columns=clip_cols), pd.DataFrame(pv_t, columns=text_cols)], axis=1)Xs = pd.concat([Xs, pd.DataFrame(ps_c, columns=clip_cols), pd.DataFrame(ps_t, columns=text_cols)], axis=1)print(f'Xt fusion shape: {Xt.shape}  (expected ~92 columns: 42 tabular + 25 CLIP-PCA + 25 text-PCA)')

In [ ]:
# ---- Train Fusion CatBoost ----------------------------------------------model = CatBoostClassifier(**CATBOOST_BASE)t0 = time.time()model.fit(Xt, y_train, cat_features=MY_CATS, eval_set=(Xv, y_val), use_best_model=True)dt = time.time() - t0print(f'\nbest_iter={model.best_iteration_}, fit={dt:.1f}s')proba_fusion_new = model.predict_proba(Xs)[:, 1].astype('float32')

In [ ]:
# ---- Save model as ftmff_catboost.cbm -----------------------------------out_cbm = ARTIFACTS / 'ftmff_catboost.cbm'model.save_model(str(out_cbm), format='cbm')print(f'Saved model: {out_cbm}  ({out_cbm.stat().st_size/1024:.1f} KB)')# Also save the PCA fitted objects in case Diana needs to reconstruct features at inferenceimport joblibjoblib.dump(pca_c, ARTIFACTS / 'ftmff_pca_clip.pkl')joblib.dump(pca_t, ARTIFACTS / 'ftmff_pca_text.pkl')print(f'Saved PCAs: {ARTIFACTS / "ftmff_pca_clip.pkl"}, ftmff_pca_text.pkl')

In [ ]:
# ---- Sanity check: compare new probas with the canonical file -----------from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curveproba_old = np.load(PROBA / 'test_proba_fusion_team.npy')def metrics(y, p):    roc = roc_auc_score(y, p)    pr = average_precision_score(y, p)    prec, rc, _ = precision_recall_curve(y, p)    m = prec[:-1] >= 0.9    rap = float(rc[:-1][m].max()) if m.any() else 0.0    return roc, pr, rapold_roc, old_pr, old_rap = metrics(y_test, proba_old)new_roc, new_pr, new_rap = metrics(y_test, proba_fusion_new)print('=== Comparison: canonical (Colab) vs newly trained (LOCAL) ===')print(f'  Canonical:   ROC={old_roc:.4f}  PR={old_pr:.4f}  R@P90={old_rap:.4f}')print(f'  New (local): ROC={new_roc:.4f}  PR={new_pr:.4f}  R@P90={new_rap:.4f}')print()corr = float(np.corrcoef(proba_old, proba_fusion_new)[0, 1])print(f'Pearson corr(old, new) = {corr:.6f}')print(f'Max abs diff = {np.max(np.abs(proba_old - proba_fusion_new)):.6f}')print(f'Mean abs diff = {np.mean(np.abs(proba_old - proba_fusion_new)):.6f}')print()if abs(old_pr - new_pr) < 0.005 and abs(old_rap - new_rap) < 0.01:    print('✅ Метрики совпадают в пределах CPU/GPU численного шума. Модель валидна для Дианы.')else:    print('⚠️ Метрики расходятся больше ожидаемого. Проверить feature alignment.')

In [ ]:
# ---- Save probas + summary for cross-reference --------------------------np.save(ARTIFACTS / 'test_proba_fusion_team_LOCAL_retrain.npy', proba_fusion_new)summary = {    'context': 'Local retrain of Fusion_team for ftmff_catboost.cbm',    'date': time.strftime('%Y-%m-%d'),    'best_iteration': int(model.best_iteration_),    'fit_seconds': round(dt, 1),    'spw': spw,    'feature_count': int(Xt.shape[1]),    'metrics_new_local': {'roc': new_roc, 'pr_auc': new_pr, 'r_at_p90': new_rap},    'metrics_canonical': {'roc': old_roc, 'pr_auc': old_pr, 'r_at_p90': old_rap},    'probas_pearson_corr_with_canonical': corr,    'probas_max_abs_diff': float(np.max(np.abs(proba_old - proba_fusion_new))),    'artifacts': {        'model': 'team_runs/artifacts/ftmff_catboost.cbm',        'pca_clip': 'team_runs/artifacts/ftmff_pca_clip.pkl',        'pca_text': 'team_runs/artifacts/ftmff_pca_text.pkl',        'probas_local': 'team_runs/artifacts/test_proba_fusion_team_LOCAL_retrain.npy',    }}with open(ARTIFACTS / 'ftmff_catboost_summary.json', 'w') as f:    json.dump(summary, f, indent=2)print(json.dumps(summary, indent=2, ensure_ascii=False))